In [13]:
!pip install -U transformers


In [14]:
pip install -U datasets


In [16]:
from datasets import load_dataset
import pandas as pd

# KOLD 데이터셋 로드 (훈련셋 1000개 샘플만 우선 확인)
dataset = load_dataset("nayohan/KOLD", split="train[:1000]")

# DataFrame으로 변환
df = pd.DataFrame(dataset)

# 상위 샘플 확인
print(df.head())
print(df.columns)


            guid      source       date                              title  \
0  kold-v1_00000  naver_news 2021-07-25            페미니즘이 범죄가 되는 나라 [삶과 문화]   
1  kold-v1_00001  naver_news 2021-04-13        [젠더의학①] 여성은 ‘몸집 작은 남성’이 아니다   
2  kold-v1_00002  naver_news 2021-09-15  “책 생명 늘려야죠”… 문학 속 ‘성차별’ 패치 떼는 출판계   
3  kold-v1_00003  naver_news 2021-01-14  이루다로 촉발된 젠더 논쟁... 개인정보 유출이란 본질 외면   
4  kold-v1_00004  naver_news 2021-05-06        GS25, 브레이브걸스 포스터 또 젠더 이슈 논란   

                                             comment   OFF         TGT  \
0  남녀평등 주장할 거면 여성징병제에도 동의하라고ㅋㅋㅋ 그리고 내 말에 그냥 시비만 걸...  True       group   
1                                 의학에도 젠더 사상이 붙네 ㄷㄷ;  True  untargeted   
2  루브르 박물관에 있는 모나리자 머리도 단발로 수정하고,미국에 있는 자유의 여신상도 ...  True       group   
3  진짜 어이가 없네 딥페이크 만든 사람들도 처벌하고 알페스 만든 사람들도 처벌하라니깐...  True  untargeted   
4                                       브레이브걸스=페미아이돌  True       group   

               GRP                                           OFF_span  \
0  others-fem

In [17]:
# 1. comment와 OFF 컬럼만 추출
df = df[["comment", "OFF"]]

# 2. OFF 값을 0/1로 변환 (욕설이면 1)
df["label"] = df["OFF"].astype(int)

# 3. 결측값 제거 (예방 차원)
df = df.dropna(subset=["comment", "label"]).reset_index(drop=True)

# 4. 라벨 분포 확인
print(df["label"].value_counts())


label
0    503
1    497
Name: count, dtype: int64


<ipython-input-17-d523beda0a63>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["label"] = df["OFF"].astype(int)


In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [19]:
# 1. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

# 2. 분류용 BERT 모델 로드 (이진 분류니까 num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained(
    "klue/bert-base",
    num_labels=2
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
from sklearn.model_selection import train_test_split
from transformers import TrainingArguments, Trainer
import torch


In [22]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["comment"], df["label"],
    test_size=0.1,
    stratify=df["label"],  # ✅ 욕설 라벨 비율 유지
    random_state=42
)


In [23]:
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True)


In [24]:
class KOLDDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = KOLDDataset(train_encodings, list(train_labels))
val_dataset = KOLDDataset(val_encodings, list(val_labels))


In [25]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,  # 테스트용으로 2 에포크
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [26]:
trainer.train()


Step,Training Loss


TrainOutput(global_step=450, training_loss=0.4210474650065104, metrics={'train_runtime': 2366.0527, 'train_samples_per_second': 0.761, 'train_steps_per_second': 0.19, 'total_flos': 84174982164000.0, 'train_loss': 0.4210474650065104, 'epoch': 2.0})

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }

# Trainer에 compute_metrics 붙이기
trainer.compute_metrics = compute_metrics

# 평가 실행
metrics = trainer.evaluate()
print("📊 평가 결과:", metrics)


📊 평가 결과: {'eval_loss': 1.2373374700546265, 'eval_accuracy': 0.74, 'eval_precision': 0.8, 'eval_recall': 0.64, 'eval_f1': 0.7111111111111111, 'eval_runtime': 48.0981, 'eval_samples_per_second': 2.079, 'eval_steps_per_second': 0.52, 'epoch': 2.0}


In [96]:
import torch
import requests
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1) 욕설 사전 로드
def load_bad_words(url):
    res = requests.get(url)
    res.raise_for_status()
    return res.json()['words']

bad_words_url = "https://cdn.jsdelivr.net/gh/hlog2e/bad_word_list@master/word_list.json"
bad_words = load_bad_words(bad_words_url)

# 2) 욕설 마스킹 함수 (정규표현식 이용)
def mask_bad_words(text, bad_words):
    escaped_words = [re.escape(word) for word in bad_words if word.strip() != '']
    pattern = re.compile('|'.join(escaped_words), flags=re.IGNORECASE)

    def replacer(match):
        return '*' * len(match.group())

    return pattern.sub(replacer, text)

# 3) 토크나이저 및 모델 로드 (이미 존재하면 재사용)
try:
    tokenizer
except NameError:
    tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
try:
    model
except NameError:
    model = AutoModelForSequenceClassification.from_pretrained("./results")
model.eval()

# 4) 욕설 확률 및 마스킹 함수
def predict_and_mask_with_prob(text, model, tokenizer, bad_words, threshold=0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        off_prob = probs[0][1].item()  # 욕설 클래스 확률

    # 욕설 확률 리턴은 항상 함꼐 (필요시 UI 표시용)
    if off_prob >= threshold:
        masked_text = mask_bad_words(text, bad_words)
        return masked_text, off_prob
    else:
        return text, off_prob

# 5) 테스트
test_text = "저새끼 진짜 너무하네."
masked_text, prob = predict_and_mask_with_prob(test_text, model, tokenizer, bad_words, threshold=0.5)
print(f"욕설 확률: {prob:.3f}")
print("마스킹된 문장:", masked_text)

##근데 이건 거의 JSON기반으로만 됨 왜했지.?

욕설 확률: 0.991
마스킹된 문장: *** 진짜 너무하네.


분리


In [76]:
pip install sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 632.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [97]:
import torch
import re
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import requests

# 1) 욕설 사전 로드
def load_bad_words(url):
    res = requests.get(url)
    res.raise_for_status()
    return res.json()['words']

bad_words_url = "https://cdn.jsdelivr.net/gh/hlog2e/bad_word_list@master/word_list.json"
bad_words = load_bad_words(bad_words_url)

# 2) 문장 임베딩 모델 (유사 단어 탐색용)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
bad_words_emb = embedding_model.encode(bad_words, convert_to_tensor=True)

# 3) 기본 욕설 마스킹 함수
def mask_words(text, words_to_mask):
    escaped = [re.escape(w) for w in words_to_mask if w.strip()]
    pattern = re.compile('|'.join(escaped), flags=re.IGNORECASE)
    return pattern.sub(lambda m: '*' * len(m.group()), text)

# 4) 모델, 토크나이저 로드(생략 가능: 이미 로드돼 있다면)
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
model.eval()

# 5) 확장 단어 탐색 함수: 문장 단어별 욕설 사전 유사 단어 포함 여부 판단
def find_similar_bad_words(words, bad_words, bad_words_emb, threshold=0.7):
    # words: 문장 내 단어 리스트
    # bad_words_emb: 욕설 사전 임베딩 텐서
    words_emb = embedding_model.encode(words, convert_to_tensor=True)
    cos_scores = util.cos_sim(words_emb, bad_words_emb)  # (len(words), len(bad_words))
    similar_words = set()
    for i, word in enumerate(words):
        max_score = torch.max(cos_scores[i]).item()
        if max_score >= threshold:
            similar_words.add(word)
    return similar_words

# 6) 통합 욕설 예측 및 마스킹 함수
def predict_and_mask_expanded(text, model, tokenizer, bad_words, bad_words_emb, threshold_model=0.5, threshold_sim=0.7):
    # 문장 욕설 확률 예측
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        off_prob = probs[0][1].item()

    # 문장 내 단어 리스트 (간단 토큰화)
    words = re.findall(r'\w+', text)

    # 욕설 문장 기준 이상이고, 욕설 사전 또는 유사 단어 마스킹
    if off_prob >= threshold_model:
        # 사전 기반 욕설 단어
        words_to_mask = set(word for word in words if any(bw.lower() == word.lower() for bw in bad_words))
        # 임베딩 유사 단어 추가
        similar_words = find_similar_bad_words(words, bad_words, bad_words_emb, threshold=threshold_sim)
        words_to_mask.update(similar_words)

        # 마스킹 적용
        masked_text = mask_words(text, words_to_mask)
        return masked_text, off_prob
    else:
        return text, off_prob




In [158]:
# 7) 테스트
test_sentence = "니같은건필요없어"
masked_text, prob = predict_and_mask_expanded(test_sentence, model, tokenizer, bad_words, bad_words_emb)
print(f"욕설 확률: {prob:.3f}")
print("마스킹된 문장:", masked_text)

욕설 확률: 0.998
마스킹된 문장: ********


In [90]:
# 학습 완료 후 저장
trainer.save_model("./content/drive/MyDrive/텍마")  # 모델과 토크나이저 모두 저장됨
tokenizer.save_pretrained("./content/drive/MyDrive/텍마")


('./content/drive/MyDrive/텍마/tokenizer_config.json',
 './content/drive/MyDrive/텍마/special_tokens_map.json',
 './content/drive/MyDrive/텍마/vocab.txt',
 './content/drive/MyDrive/텍마/added_tokens.json',
 './content/drive/MyDrive/텍마/tokenizer.json')

In [89]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
